In [0]:
from dlt import table
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

# ✅ Define schema for retail IoT CSV
iot_schema = StructType([
    StructField("OrderID", StringType(), True),
    StructField("CustomerID", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("StoreID", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("SalesAmount", DoubleType(), True),
    StructField("OrderTime", StringType(), True),       # can cast to Timestamp later in Silver
    StructField("DeviceType", StringType(), True),
    StructField("Temperature", DoubleType(), True),
    StructField("DeviceStatus", StringType(), True),
    StructField("AnomalyFlag", IntegerType(), True),
    StructField("AnomalyType", StringType(), True)
])

# ✅ Paths (use subdirectories to avoid overlap with DLT system files)
raw_path = "abfss://raw01hetul@hetulstorage.dfs.core.windows.net/raw_data/"
schema_path = "abfss://raw01hetul@hetulstorage.dfs.core.windows.net/dlt_schemas/bronze_orders_hetul"
checkpoint_path = "abfss://raw01hetul@hetulstorage.dfs.core.windows.net/dlt_checkpoints/bronze_orders_hetul"

@table(
    name="hetul_project01_catalog.capstone_retail_hetul.bronze_orders",
    comment="Bronze table - raw ingested IoT retail data via Auto Loader"
)
def bronze_orders():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", True)
            .option("cloudFiles.schemaLocation", schema_path)
            .schema(iot_schema)   # ✅ manually set schema (no infer needed)
            .load(raw_path)       # ✅ points to /raw_data/ folder
    )


In [0]:
from dlt import table
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType, IntegerType

# =========================================
# Silver Layer - Cleaned IoT Orders
# =========================================
@table(
    name="capstone_retail_hetul.silver_orders",
    comment="Silver layer: cleaned and deduplicated IoT orders with anomaly handling"
)
def silver_orders():
    # Read Bronze
    orders = dlt.read("bronze_orders")

    # Standardize schema
    orders = (
        orders.withColumn("Quantity", F.col("Quantity").cast(IntegerType()))
              .withColumn("UnitPrice", F.col("UnitPrice").cast(DoubleType()))
              .withColumn("SalesAmount", F.col("SalesAmount").cast(DoubleType()))
              .withColumn("Temperature", F.col("Temperature").cast(DoubleType()))
              .withColumn("OrderTime", F.to_timestamp("OrderTime", "M/d/yyyy H:mm"))
    )

    # Deduplicate by OrderID
    orders = orders.dropDuplicates(["OrderID"])

    # Handle missing anomaly type
    orders = orders.withColumn(
        "AnomalyType",
        F.when((F.col("AnomalyFlag") == 1) & (F.col("AnomalyType").isNull()), "Unknown")
         .when((F.col("AnomalyFlag") == 1) & (F.col("AnomalyType") == ""), "Unknown")
         .otherwise(F.col("AnomalyType"))
    )

    # Add business rule fields
    orders = (
        orders.withColumn("IsAnomaly", F.col("AnomalyFlag") == 1)
              .withColumn("DeviceStatus", F.upper(F.col("DeviceStatus")))
              .withColumn("NormalizedStoreID", F.upper(F.col("StoreID")))
              .withColumn("NormalizedProduct", F.upper(F.col("Product")))
    )

    return orders


In [0]:
from dlt import table, read
import pyspark.sql.functions as F

# ===============================
# Gold Layer Tables (capstone_retail.gold)
# ===============================

# 1. Daily / Monthly Sales per Region
@table(
    name="hetul_project01_catalog.capstone_retail_hetul.gold_sales_region",
    comment="Daily & Monthly Sales per Region"
)
def gold_sales_region():
    return (
        read("hetul_project01_catalog.capstone_retail_hetul.silver_orders")
        .withColumn("OrderDate", F.to_date("OrderTime", "M/d/yyyy H:mm"))
        .withColumn("YearMonth", F.date_format("OrderDate", "yyyy-MM"))
        .groupBy("Region", "OrderDate", "YearMonth")
        .agg(F.sum("SalesAmount").alias("TotalSales"))
    )

# 2. Device Anomaly Trend per Store
@table(
    name="hetul_project01_catalog.capstone_retail_hetul.gold_anomaly_trends",
    comment="Device Anomaly Trend per Store"
)
def gold_anomaly_trends():
    return (
        read("hetul_project01_catalog.capstone_retail_hetul.silver_orders")
        .withColumn("OrderDate", F.to_date("OrderTime", "M/d/yyyy H:mm"))
        .groupBy("StoreID", "OrderDate", "DeviceType", "AnomalyType")
        .agg(F.count("*").alias("AnomalyCount"))
    )

# 3. Conversion Rate Impact when devices fail
@table(
    name="hetul_project01_catalog.capstone_retail_hetul.gold_conversion_impact",
    comment="Conversion rate impact of device failures"
)
def gold_conversion_impact():
    df = read("hetul_project01_catalog.capstone_retail_hetul.silver_orders")
    total_orders = df.groupBy("StoreID").agg(F.count("*").alias("TotalOrders"))
    failed_orders = (
        df.filter(df.AnomalyFlag == 1)
          .groupBy("StoreID")
          .agg(F.count("*").alias("FailedOrders"))
    )
    return (
        total_orders.join(failed_orders, "StoreID", "left")
        .withColumn("FailedOrders", F.coalesce(F.col("FailedOrders"), F.lit(0)))
        .withColumn("FailureRate", F.col("FailedOrders") / F.col("TotalOrders"))
    )

# 4. Store Tier Classification
@table(
    name="hetul_project01_catalog.capstone_retail_hetul.gold_store_tiers",
    comment="Store tiers based on total sales (High / Medium / Low performers)"
)
def gold_store_tiers():
    df = (
        read("hetul_project01_catalog.capstone_retail_hetul.silver_orders")
        .groupBy("StoreID")
        .agg(F.sum("SalesAmount").alias("TotalSales"))
    )

    return (
        df.withColumn(
            "StoreTier",
            F.when(F.col("TotalSales") > 100000, "High")
             .when(F.col("TotalSales") > 50000, "Medium")
             .otherwise("Low")
        )
    )

# 5. Weighted Average Sales vs Anomalies
@table(
    name="hetul_project01_catalog.capstone_retail_hetul.gold_weighted_sales_anomalies",
    comment="Weighted average of sales vs anomalies per store"
)
def gold_weighted_sales_anomalies():
    df = read("hetul_project01_catalog.capstone_retail_hetul.silver_orders")
    return (
        df.groupBy("StoreID")
        .agg(
            F.sum("SalesAmount").alias("TotalSales"),
            F.sum(F.when(F.col("AnomalyFlag") == 1, 1).otherwise(0)).alias("TotalAnomalies")
        )
        .withColumn("SalesPerAnomaly", 
            F.when(F.col("TotalAnomalies") > 0, F.col("TotalSales") / F.col("TotalAnomalies"))
             .otherwise(F.lit(None))
        )
    )
